In [3]:
import sys
from pathlib import Path

def is_google_colab() -> bool:
    if "google.colab" in str(get_ipython()):
        return True
    return False

def clone_repository() -> None:
    !git clone https://github.com/featurestorebook/mlfs-book.git
    %cd mlfs-book

def install_dependencies() -> None:
    !pip install --upgrade uv
    !uv pip install --all-extras --system --requirement pyproject.toml

if is_google_colab():
    clone_repository()
    install_dependencies()
    root_dir = str(Path().absolute())
    print("Google Colab environment")
else:
    root_dir = Path().absolute()
    # Strip ~/notebooks/ccfraud from PYTHON_PATH if notebook started in one of these subdirectories
    if root_dir.parts[-1:] == ('airquality',):
        root_dir = Path(*root_dir.parts[:-1])
    if root_dir.parts[-1:] == ('notebooks',):
        root_dir = Path(*root_dir.parts[:-1])
    root_dir = str(root_dir) 
    print("Local environment")

# Add the root directory to the `PYTHONPATH` to use the `recsys` Python module from the notebook.
if root_dir not in sys.path:
    sys.path.append(root_dir)
print(f"Added the following directory to the PYTHONPATH: {root_dir}")
    
# Set the environment variables from the file <root_dir>/.env
from mlfs import config
settings = config.HopsworksSettings(_env_file=f"{root_dir}/.env")

Local environment
Added the following directory to the PYTHONPATH: /Users/jesper/Documents/KTH/Skalbar_ML/mlfs-book
HopsworksSettings initialized!


<span style="font-width:bold; font-size: 3rem; color:#333;">- Part 02: Daily Feature Pipeline for Air Quality (aqicn.org) and weather (openmeteo)</span>

## 🗒️ This notebook is divided into the following sections:
1. Download and Parse Data
2. Feature Group Insertion


__This notebook should be scheduled to run daily__

In the book, we use a GitHub Action stored here:
[.github/workflows/air-quality-daily.yml](https://github.com/featurestorebook/mlfs-book/blob/main/.github/workflows/air-quality-daily.yml)

However, you are free to use any Python Orchestration tool to schedule this program to run daily.

### <span style='color:#ff5f27'> 📝 Imports

In [4]:
import datetime
import time
import requests
import pandas as pd
import hopsworks
from mlfs.airquality import util
from mlfs import config
import json
import os
import warnings
warnings.filterwarnings("ignore")

## <span style='color:#ff5f27'> 🌍 Get the Sensor URL, Country, City, Street names from Hopsworks </span>

__Update the values in the cell below.__

__These should be the same values as in notebook 1 - the feature backfill notebook__


In [5]:
project = hopsworks.login(engine="python")
fs = project.get_feature_store() 
secrets = hopsworks.get_secrets_api()

# This line will fail if you have not registered the AQICN_API_KEY as a secret in Hopsworks
AQICN_API_KEY = secrets.get_secret("AQICN_API_KEY").value
location_str = secrets.get_secret("SENSOR_LOCATION_JSON").value
location = json.loads(location_str)

country=location['country']
city=location['city']
street=location['street']
aqicn_url=location['aqicn_url']
latitude=location['latitude']
longitude=location['longitude']

today = datetime.date.today()

location_str

2025-11-11 09:43:05,626 INFO: Initializing external client
2025-11-11 09:43:05,627 INFO: Base URL: https://c.app.hopsworks.ai:443


2025-11-11 09:43:07,337 INFO: Python Engine initialized.

Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/1271996


'{"country": "Sweden", "city": "visby", "street": "\\u00f6stra tv\\u00e4rgatan", "aqicn_url": "https://api.waqi.info/feed/A60076", "latitude": "57.6339483", "longitude": "18.31051"}'

### <span style="color:#ff5f27;"> 🔮 Get references to the Feature Groups </span>

In [6]:
# Retrieve feature groups
air_quality_fg = fs.get_feature_group(
    name='air_quality',
    version=1,
)
weather_fg = fs.get_feature_group(
    name='weather',
    version=1,
)

---

## <span style='color:#ff5f27'> 🌫 Retrieve Today's Air Quality data (PM2.5) from the AQI API</span>


In [7]:
import requests
import pandas as pd
import os

# Load all configured locations from secrets
sensor_secret_names = [s.strip() for s in os.getenv("SENSOR_SECRET_NAMES", "").split(",") if s.strip()]
if not sensor_secret_names:
    sensor_secret_names = ["SENSOR_LOCATION_JSON"]

locations = []
for secret_name in sensor_secret_names:
    loc_str = secrets.get_secret(secret_name).value
    locations.append(json.loads(loc_str))

# Retrieve AQ data for all configured stations
frames = []
for loc in locations:
    df = util.get_pm25(
        loc['aqicn_url'],
        loc['country'],
        loc['city'],
        loc['street'],
        today,
        AQICN_API_KEY,
    )
    frames.append(df)

aq_today_df = pd.concat(frames, ignore_index=True)
aq_today_df

,pm25,country,city,street,date,url
0,0.0,sweden,borgholm,norra långgatan,2025-11-11,https://api.waqi.info/feed/A71104
1,3.0,sweden,ljugarn,storvägen,2025-11-11,https://api.waqi.info/feed/A376954
2,33.0,sweden,visby,östra tvärgatan,2025-11-11,https://api.waqi.info/feed/A60076


In [8]:
aq_today_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   pm25     3 non-null      float32       
 1   country  3 non-null      object        
 2   city     3 non-null      object        
 3   street   3 non-null      object        
 4   date     3 non-null      datetime64[ns]
 5   url      3 non-null      object        
dtypes: datetime64[ns](1), float32(1), object(4)
memory usage: 264.0+ bytes


## <span style='color:#ff5f27'> 🌦 Get Weather Forecast data</span>

In [9]:
# Retrieve weather forecasts for all configured stations and keep the midday sample per day
weather_frames = []
for loc in locations:
    hourly_df = util.get_hourly_weather_forecast(loc['city'], loc['latitude'], loc['longitude'])
    hourly_df = hourly_df.set_index('date')
    # Only get the daily weather data around noon
    daily_df = hourly_df.between_time('11:59', '12:01')
    daily_df = daily_df.reset_index()
    daily_df['date'] = pd.to_datetime(daily_df['date']).dt.date
    daily_df['date'] = pd.to_datetime(daily_df['date'])
    daily_df['city'] = loc['city']
    weather_frames.append(daily_df)

daily_all_df = pd.concat(weather_frames, ignore_index=True)
daily_all_df

Coordinates 57.25°N 17.0°E
Elevation 5.0 m asl
Timezone None None
Timezone difference to GMT+0 0 s
Coordinates 57.5°N 18.75°E
Elevation 11.0 m asl
Timezone None None
Timezone difference to GMT+0 0 s
Coordinates 57.5°N 18.5°E
Elevation 48.0 m asl
Timezone None None
Timezone difference to GMT+0 0 s


,date,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,wind_direction_10m_dominant,city
0,2025-11-11,9.05,0.0,5.154416,192.094742,borgholm
1,2025-11-12,10.25,0.0,26.544964,200.647064,borgholm
2,2025-11-13,10.45,0.2,17.819090,224.999893,borgholm
3,2025-11-14,5.45,0.0,12.599998,306.869965,borgholm
4,2025-11-15,3.90,0.0,12.240000,270.000000,borgholm
5,2025-11-16,5.15,0.0,19.093580,224.236191,borgholm
6,2025-11-17,3.95,0.0,9.449572,287.744751,borgholm
7,2025-11-11,8.90,0.0,5.014219,201.037582,ljugarn
8,2025-11-12,10.70,0.0,24.514387,209.953522,ljugarn
9,2025-11-13,10.05,0.1,16.375053,236.659271,ljugarn


In [10]:
daily_all_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 6 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   date                         21 non-null     datetime64[ns]
 1   temperature_2m_mean          21 non-null     float32       
 2   precipitation_sum            21 non-null     float32       
 3   wind_speed_10m_max           21 non-null     float32       
 4   wind_direction_10m_dominant  21 non-null     float32       
 5   city                         21 non-null     object        
dtypes: datetime64[ns](1), float32(4), object(1)
memory usage: 804.0+ bytes


## <span style="color:#ff5f27;">⬆️ Uploading new data to the Feature Store</span>

In [12]:
# Compute pm25_roll3 per station using recent history, then insert
import numpy as np

# Ensure types and tz-naive dates
aq_today_df['pm25'] = aq_today_df['pm25'].astype('float32')
aq_today_df['date'] = pd.to_datetime(aq_today_df['date']).dt.tz_localize(None)
aq_today_df['pm25_roll3'] = pd.Series(np.nan, index=aq_today_df.index, dtype='float32')

# Read historical data once and filter locally
hist_all = air_quality_fg.read()
if len(hist_all) > 0:
    hist_all = hist_all[['country','city','street','date','pm25']].copy()
    hist_all['date'] = pd.to_datetime(hist_all['date']).dt.tz_localize(None)
else:
    hist_all = pd.DataFrame(columns=['country','city','street','date','pm25'])

for (country, city, street), df_new in aq_today_df.groupby(['country','city','street']):
    hist = hist_all[(hist_all['country'] == country) &
                    (hist_all['city'] == city) &
                    (hist_all['street'] == street)][['date','pm25']]
    if len(hist) > 0:
        hist = hist.sort_values('date').tail(5)
    else:
        hist = pd.DataFrame(columns=['date','pm25'])

    combined = pd.concat([hist, df_new[['date','pm25']]], ignore_index=True).sort_values('date')
    combined['pm25_roll3'] = combined['pm25'].rolling(window=3, min_periods=1).mean().shift(1).astype('float32')

    computed = combined[combined['date'].isin(df_new['date'])][['date','pm25_roll3']]
    for _, row in computed.iterrows():
        mask = (
            (aq_today_df['country'] == country) &
            (aq_today_df['city'] == city) &
            (aq_today_df['street'] == street) &
            (aq_today_df['date'] == row['date'])
        )
        aq_today_df.loc[mask, 'pm25_roll3'] = row['pm25_roll3']

# Final dtypes
aq_today_df['pm25_roll3'] = pd.to_numeric(aq_today_df['pm25_roll3'], errors='coerce').astype('float32')

# Insert new data
air_quality_fg.insert(aq_today_df)

Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.52s) 
2025-11-11 09:47:46,701 INFO: 	1 expectation(s) included in expectation_suite.
Validation succeeded.
Validation Report saved successfully, explore a summary at https://c.app.hopsworks.ai:443/p/1271996/fs/1258596/fg/1638098


Uploading Dataframe: 100.00% |██████████| Rows 3/3 | Elapsed Time: 00:00 | Remaining Time: 00:00


Launching job: air_quality_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://c.app.hopsworks.ai:443/p/1271996/jobs/named/air_quality_1_offline_fg_materialization/executions


(Job('air_quality_1_offline_fg_materialization', 'SPARK'),
 {
   "success": true,
   "results": [
     {
       "success": true,
       "expectation_config": {
         "expectation_type": "expect_column_min_to_be_between",
         "kwargs": {
           "column": "pm25",
           "min_value": -0.1,
           "max_value": 500.0,
           "strict_min": true
         },
         "meta": {
           "expectationId": 735369
         }
       },
       "result": {
         "observed_value": 0.0,
         "element_count": 3,
         "missing_count": null,
         "missing_percent": null
       },
       "meta": {
         "ingestionResult": "INGESTED",
         "validationTime": "2025-11-11T08:47:46.000700Z"
       },
       "exception_info": {
         "raised_exception": false,
         "exception_message": null,
         "exception_traceback": null
       }
     }
   ],
   "evaluation_parameters": {},
   "statistics": {
     "evaluated_expectations": 1,
     "successful_expectati

In [13]:
# Insert weather for all stations
weather_fg.insert(daily_all_df, wait=True)

2025-11-11 09:48:14,148 INFO: 	2 expectation(s) included in expectation_suite.
Validation succeeded.
Validation Report saved successfully, explore a summary at https://c.app.hopsworks.ai:443/p/1271996/fs/1258596/fg/1638099


Uploading Dataframe: 100.00% |██████████| Rows 21/21 | Elapsed Time: 00:00 | Remaining Time: 00:00


Launching job: weather_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://c.app.hopsworks.ai:443/p/1271996/jobs/named/weather_1_offline_fg_materialization/executions
2025-11-11 09:48:30,356 INFO: Waiting for execution to finish. Current state: SUBMITTED. Final status: UNDEFINED
2025-11-11 09:48:33,522 INFO: Waiting for execution to finish. Current state: RUNNING. Final status: UNDEFINED
2025-11-11 09:50:09,167 INFO: Waiting for log aggregation to finish.
2025-11-11 09:50:17,793 INFO: Execution finished successfully.


(Job('weather_1_offline_fg_materialization', 'SPARK'),
 {
   "success": true,
   "results": [
     {
       "success": true,
       "expectation_config": {
         "expectation_type": "expect_column_min_to_be_between",
         "kwargs": {
           "column": "wind_speed_10m_max",
           "min_value": -0.1,
           "max_value": 1000.0,
           "strict_min": true
         },
         "meta": {
           "expectationId": 735371
         }
       },
       "result": {
         "observed_value": 4.510787487030029,
         "element_count": 21,
         "missing_count": null,
         "missing_percent": null
       },
       "meta": {
         "ingestionResult": "INGESTED",
         "validationTime": "2025-11-11T08:48:14.000148Z"
       },
       "exception_info": {
         "raised_exception": false,
         "exception_message": null,
         "exception_traceback": null
       }
     },
     {
       "success": true,
       "expectation_config": {
         "expectation_type":

## <span style="color:#ff5f27;">⏭️ **Next:** Part 03: Training Pipeline
 </span> 

In the following notebook you will read from a feature group and create training dataset within the feature store
